# Set Up:

In [1]:
%pip install rdflib
%pip install folium
%pip install shapely
%pip install pandas
%pip install plotly

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
from pathlib import Path
from rdflib import Namespace, URIRef, BNode, Literal, ConjunctiveGraph
from rdflib.namespace import RDF, split_uri
from collections import defaultdict
import folium
from shapely import wkt
from shapely.geometry import mapping
from IPython.display import display
import branca.colormap as cm
import pandas as pd
import ipywidgets as widgets
import plotly.express as px
import time
import plotly.io as pio
pio.renderers.default = 'iframe'

In [3]:
saref = Namespace("https://saref.etsi.org/core/")
s4ener = Namespace("https://saref.etsi.org/saref4ener/")
s4bldg = Namespace("https://saref.etsi.org/saref4bldg/")
geo = Namespace("http://www.opengis.net/ont/geosparql#")
ex = Namespace("https://example.org/")

# Knowledge Graph Enrichment:

In [4]:
g = ConjunctiveGraph()

folder = Path("data")

for file in folder.glob("*.log"):
    start = time.perf_counter()
    g.parse(file, format="trig")
    elapsed = time.perf_counter() - start
    print("file done", elapsed)

C:\Users\gin\AppData\Local\Temp\ipykernel_21100\762638814.py:1: DeprecationWarning: ConjunctiveGraph is deprecated, use Dataset instead.
  g = ConjunctiveGraph()


file done 20.922173099941574


In [5]:
print("Statements in total:", len(g))

Statements in total: 642323


In [6]:
print("Number of graphs:", len(list(g.contexts())))

Number of graphs: 1


In [7]:
with open("meter_locations.csv", newline="", encoding="utf-8-sig") as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        meter_uri = row["meter_uri"]
        meter_long = row["meter_long"].strip()
        meter_lat = row["meter_lat"].strip()

        meter = URIRef(meter_uri)
        meter_geom = BNode()

        g.add((meter, RDF.type, saref.Meter))
        g.add((meter, geo.hasGeometry, meter_geom))

        if meter_long and meter_lat:
            meter_long = float(meter_long)
            meter_lat = float(meter_lat)

            coord = f"POINT({meter_long} {meter_lat})"
            g.add((meter_geom, geo.asWKT, Literal(coord)))

        bldgs = row["metered_location"].split("/")

        for b in bldgs:
            if b != "":
                bldg = URIRef(ex[b.strip()])
                g.add((meter, ex.measuresBuilding, bldg))

In [8]:
with open("kadaster_buildings.csv", newline="", encoding="utf-8-sig") as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        gebouw = row["gebouw"]
        build = row["build"]

        bldg = ex[build]
        bldg_geom = BNode()

        g.add((bldg, RDF.type, s4bldg.Building))
        g.add((bldg, geo.hasGeometry, bldg_geom))
        g.add((bldg_geom, geo.asWKT, Literal(gebouw)))

# Competency Questions:

### 1. Which buildings does meter X measure?

In [9]:
# Which buildings does meter X measure?

q = """
PREFIX saref: <https://saref.etsi.org/core/>
PREFIX ex: <https://example.org/>

SELECT ?meter ?coord ?building ?polygon WHERE {
  ?meter a saref:Meter ;
         ex:measuresBuilding ?building ;
         geo:hasGeometry ?geom .
  ?geom geo:asWKT ?coord .
  ?building geo:hasGeometry ?bgeom .
  ?bgeom geo:asWKT ?polygon .
}
"""

meter_data = defaultdict(lambda: {"coord": None, "buildings": []})

for meter_uri, meter_coord, bldg_uri, bldg_poly in g.query(q):
    meter_name = meter_uri.split("meter/")[1]
    _, bldg_name = split_uri(bldg_uri)
    meter_data[meter_name]["coord"] = meter_coord
    meter_data[meter_name]["buildings"].append((bldg_name, bldg_poly))

dropdown = widgets.Dropdown(options=sorted(meter_data.keys()), description="Meter:")
output_meter = widgets.Output()

def show_meter(change):
    with output_meter:
        output_meter.clear_output()
        meter = change["new"]
        data = meter_data[meter]

        print(f"Buildings measured by {meter}:")
        for bldg_name, _ in data["buildings"]:
            print("-", bldg_name)

        m = folium.Map(location=[51.985213, 5.870865], zoom_start=17)

        meter_point = wkt.loads(data["coord"])
        folium.GeoJson(data=mapping(meter_point), tooltip=meter).add_to(m)

        for bldg_name, bldg_poly in data["buildings"]:
            bldg_shape = wkt.loads(str(bldg_poly))
            folium.GeoJson(data=mapping(bldg_shape), tooltip=bldg_name).add_to(m)

        display(m)

dropdown.observe(show_meter, names="value")
display(dropdown, output_meter)
show_meter({"new": dropdown.value})

Dropdown(description='Meter:', options=('1', '2', '3'), value='1')

Output()

### 2. Is there a meter that does not measure any building?

In [10]:
# Is there a meter that does not measure any building?

q = """
PREFIX saref: <https://saref.etsi.org/core/>
PREFIX ex: <https://example.org/>

SELECT ?meter WHERE {
  ?meter a saref:Meter .
  FILTER NOT EXISTS {
    ?meter ex:measuresBuilding ?building .
  }
}
"""

m = folium.Map(location=[51.985213, 5.870865], zoom_start=17)
meters = []

for (meter_uri,) in g.query(q):
    meter_name = meter_uri.split("meter/")[1]

    q2 = f"""
    PREFIX geo: <http://www.opengis.net/ont/geosparql#>

    SELECT ?point WHERE{{
      <{meter_uri}> geo:hasGeometry ?geometry.
      ?geometry geo:asWKT ?point.
    }}
    """

    meters.append(meter_name)

    result = g.query(q2)

    if result:
        for (location,) in result:
            coords = wkt.loads(location)

            folium.GeoJson(data=mapping(coords), tooltip=meter_name).add_to(m)

print("Meters that do not measure building usage:")
print(", ".join(meters), "\n")
print("Map with such meters that have a known location:")
m

Meters that do not measure building usage:
4, 5 

Map with such meters that have a known location:


### 3. Is there any building that has no meter measuring its usage?

In [11]:
# Is there any building that has no meter measuring its usage?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>
PREFIX ex: <https://example.org/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
  FILTER NOT EXISTS {
    ?meter ex:measuresBuilding ?building
  }

}
"""

m = folium.Map(location=[51.985213, 5.870865], zoom_start=17)
bldgs = []

for (bldg_uri,) in g.query(q):
    _, bldg_name = split_uri(bldg_uri)

    q2 = f"""
    PREFIX geo: <http://www.opengis.net/ont/geosparql#>

    SELECT ?polygon WHERE{{
      <{bldg_uri}> geo:hasGeometry ?geometry.
      ?geometry geo:asWKT ?polygon.
    }}
    """

    bldgs.append(bldg_name)

    result = g.query(q2)

    for (poly,) in result:
        polygon = wkt.loads(poly)

        folium.GeoJson(data=mapping(polygon), tooltip=bldg_name).add_to(m)

print("Buildings that have no meter measuring their usage:")
print(", ".join(bldgs), "\n")
print("Map with such buildings:")
m

Buildings that have no meter measuring their usage:
B50, B52, B01, B02, B04, B09, B11, B13, B16, B17, B18, B19, B20, B22, B30, B31, B40, B44, B46, B48, B14, B38, B54, M01 

Map with such buildings:


### 4. Which meters measure the same building?

In [12]:
# Which meters measure the same building?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX ex: <https://example.org/>

    SELECT ?meter WHERE{{
      ?meter ex:measuresBuilding <{bldg_uri}> .
    }}
    """

    if len(g.query(q2)) > 1:
      _, bldg_name = split_uri(bldg_uri)
      print("Measuring building",bldg_name,":")

      for (meter_uri,) in g.query(q2):
        meter_name = meter_uri.split("meter/")[1]

        print(meter_name)

Measuring building B42 :
2
3


### 5. What is the aggregated daily energy usage of building X?

In [13]:
# What is the aggregated daily energy usage of building X?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""

bldgs_kwh = defaultdict(lambda: defaultdict(float))

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX saref: <https://saref.etsi.org/core/>
    PREFIX ex: <https://example.org/>

    SELECT ?meter ?time ?value WHERE{{
      ?meter ex:measuresBuilding <{bldg_uri}> .
      ?observation saref:madeBy ?meter ;
                   saref:hasResult ?result ;
                   saref:hasTimestamp ?time .
      ?result saref:hasValue ?value ;
              saref:isMeasuredIn <http://qudt.org/vocab/unit/KiloW-HR> .
    }}
    ORDER BY ?meter ?time
    """

    result = g.query(q2)

    if result:
        meter_values = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
        _, bldg_name = split_uri(bldg_uri)

        for meter_uri, time, value in result:
            minute = time[:16]
            day = time[:10]
            meter_values[meter_uri][day][minute].append(float(value))

        complete = defaultdict(lambda: defaultdict(list))
        meter_counter = 0
        for meter_uri, days in meter_values.items():
            meter_counter += 1

            for day, times in days.items():
              for time, values in times.items():
                  if len(values) == 4:
                      complete[day][time].append(max(values))

        for day, times in complete.items():
            valid_time = []

            for time in sorted(times.keys()):
                if len(times[time]) == meter_counter:
                    valid_time.append(time)

            earliest = valid_time[0]
            latest = valid_time[-1]

            bldg_usage = sum(times[latest]) - sum(times[earliest])

            bldgs_kwh[bldg_name][day] = bldg_usage

kwh_bldg_dropdown = widgets.Dropdown(options=sorted(bldgs_kwh.keys()), description="Building:")
kwh_day_dropdown = widgets.Dropdown(options=sorted(bldgs_kwh[kwh_bldg_dropdown.value].keys()), description="Day:")
kwh_output = widgets.Output()

def show_kwh(change=None):
    with kwh_output:
        kwh_output.clear_output()
        bldg = kwh_bldg_dropdown.value
        day = kwh_day_dropdown.value
        print(f"Daily aggregated energy usage of {bldg} on {day}:")
        print("-", bldgs_kwh[bldg][day], "kWh")

def show_day(change):
    bldg = change["new"]
    kwh_day_dropdown.options = sorted(bldgs_kwh[bldg].keys())
    if kwh_day_dropdown.options:
        kwh_day_dropdown.value = kwh_day_dropdown.options[0]

kwh_bldg_dropdown.observe(show_day, names="value")
kwh_bldg_dropdown.observe(show_kwh, names="value")
kwh_day_dropdown.observe(show_kwh, names="value")
display(kwh_bldg_dropdown, kwh_day_dropdown, kwh_output)
show_kwh()

Dropdown(description='Building:', options=('B12', 'B42'), value='B12')

Dropdown(description='Day:', options=('2026-06-08',), value='2026-06-08')

Output()

### 6. Which building shows the highest average hourly power consumption over the time period?

In [14]:
# Which building shows the highest average hourly power consumption over the time period?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""
max_avg = ("", "", -1)

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX saref: <https://saref.etsi.org/core/>
    PREFIX ex: <https://example.org/>

    SELECT ?meter ?time ?value WHERE{{
      ?meter ex:measuresBuilding <{bldg_uri}> .
      ?observation saref:madeBy ?meter ;
                   saref:hasResult ?result ;
                   saref:hasTimestamp ?time .
      ?result saref:hasValue ?value ;
              saref:isMeasuredIn <http://qudt.org/vocab/unit/W> .
    }}
    ORDER BY ?meter ?time
    """

    result = g.query(q2)

    if result:
        _, bldg_name = split_uri(bldg_uri)

        meter_info = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
        for meter_uri, time, value in result:
            hour = time[:13]
            minute = time[:16]
            meter_info[meter_uri][hour][minute].append(float(value))

        hourly = defaultdict(list)
        filtered_values = defaultdict(lambda: defaultdict(list))
        meter_cnt = 0

        for meter_uri, hours in meter_info.items():
            meter_cnt += 1
            for hour, minutes in hours.items():
                for minute, values in minutes.items():
                    if len(values) == 4:
                        filtered_values[hour][minute].append(max(values))

        for hour, minutes in filtered_values.items():
            for minute, values in minutes.items():
                if len(values) == meter_cnt:
                    hourly[hour].append(sum(values))

        avge = {}
        for hour in hourly:
            avge[hour] = sum(hourly[hour]) / len(hourly[hour])

        for hour, avge in avge.items():
            if avge > max_avg[2]:
                max_avg = (bldg_name, hour, avge)

n, h, a = max_avg
print(f"The building with the highest average hourly power consumption is {n}, with a peak average of {a} W and reaching that during {h[11:]}:00 on {h[:10]}.")

The building with the highest average hourly power consumption is B12, with a peak average of 45435.63892857143 W and reaching that during 08:00 on 2026-06-08.


### 7. How do the daily power and energy usage timelines of buildings X and Y differ?

In [15]:
# How do the daily power and energy usage timelines of buildings X and Y differ?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>

SELECT ?building WHERE {
  ?building a s4bldg:Building .
}
"""

kwh_row = []
w_row = []

for (bldg_uri,) in g.query(q):
    q2 = f"""
    PREFIX saref: <https://saref.etsi.org/core/>
    PREFIX ex: <https://example.org/>
    PREFIX saref4ab: <https://linkeddata.arnhemsbuiten.nl/ontology/saref4ab/>

    SELECT ?meter ?time ?value ?property WHERE{{
      ?meter ex:measuresBuilding <{bldg_uri}> .
      ?observation saref:madeBy ?meter ;
                   saref:hasResult ?result ;
                   saref:hasTimestamp ?time .
      ?result saref:hasValue ?value ;
              saref:isMeasuredIn ?property .
    }}
    ORDER BY ?meter ?time
    """

    result = g.query(q2)

    if result:
        _, bldg_name = split_uri(bldg_uri)
        bldg_kWh = defaultdict(lambda: defaultdict(float))
        bldg_W = defaultdict(lambda: defaultdict(list))

        meter_vals = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))
        for meter_uri, time, value, prpt_uri in result:
            minute = time[:16]
            day = time[:10]
            meter_vals[meter_uri][prpt_uri][day][minute].append(float(value))

        complete = defaultdict(lambda: defaultdict(list))
        meter_counter = 0
        for meter_uri, properties in meter_vals.items():
            meter_counter += 1
            for prpt_uri, days in properties.items():
                for day, times in days.items():
                    prpt_name = prpt_uri.split("/")[-1]
                    for time, values in times.items():
                        if prpt_name == "KiloW-HR":
                            if len(values) == 4:
                                complete[day][time].append(max(values))

                        elif prpt_name == "W":
                            if len(values) == 4:
                                bldg_W[day][time].append(max(values))

        for day, times in complete.items():
            sorted_time = sorted(times.keys())

            for i in range(1, len(sorted_time)):
                prev_t = sorted_time[i-1]
                cur_t = sorted_time[i]

                if len(times[cur_t]) == meter_counter and len(times[prev_t]) == meter_counter:
                    bldg_kWh[day][cur_t] += sum(times[cur_t]) - sum(times[prev_t])

        for day, times in bldg_kWh.items():
            for time, val in times.items():
                kwh_row.append({"building": bldg_name, "day": pd.to_datetime(day), "time": pd.to_datetime(time), "kwh": val})

        for day, times in bldg_W.items():
            for time, vals in times.items():
                if len(vals) == meter_counter:
                    w_row.append({"building": bldg_name, "day": pd.to_datetime(day), "time": pd.to_datetime(time), "power": sum(vals)})

kwh_df = pd.DataFrame(kwh_row).sort_values("time")
w_df = pd.DataFrame(w_row).sort_values("time")

bldgs = sorted(w_df["building"].unique())
days = sorted(w_df["day"].dt.strftime("%Y-%m-%d").unique())

bldg_x = widgets.Dropdown(options=bldgs, description="Building X:")
bldg_y = widgets.Dropdown(options=bldgs, description="Building Y:")
day_dropdown = widgets.Dropdown(options=days, description="Day:")
output_compare = widgets.Output()

def show_graphs(change=None):
    with output_compare:
        output_compare.clear_output(wait=True)

        building_x = bldg_x.value
        building_y = bldg_y.value
        day = day_dropdown.value

        w_filtered = w_df[(w_df["building"].isin([building_x, building_y])) & (w_df["day"].dt.strftime("%Y-%m-%d") == day)]

        kwh_filtered = kwh_df[(kwh_df["building"].isin([building_x, building_y])) & (kwh_df["day"].dt.strftime("%Y-%m-%d") == day)]

        colour_map = {building_x: "royalblue", building_y: "crimson"}

        fig_w = px.line(w_filtered, x="time", y="power", color="building", color_discrete_map=colour_map, title=f"Power usage timeline on {day}", labels={"time": "Time", "power": "Power (W)", "building": "Building"})
        fig_w.show()

        fig_kwh = px.line( kwh_filtered, x="time", y="kwh", color="building", color_discrete_map=colour_map, title=f"Energy usage timeline on {day}", labels={"time": "Time", "kwh": "Energy (kWh)", "building": "Building"})
        fig_kwh.show()

bldg_x.observe(show_graphs, names="value")
bldg_y.observe(show_graphs, names="value")
day_dropdown.observe(show_graphs, names="value")
display(bldg_x, bldg_y, day_dropdown, output_compare)
show_graphs()

Dropdown(description='Building X:', options=('B12', 'B42'), value='B12')

Dropdown(description='Building Y:', options=('B12', 'B42'), value='B12')

Dropdown(description='Day:', options=('2026-06-08',), value='2026-06-08')

Output()

### 8. How can buildings be visualised spatially, using colour to indicate aggregated energy usage over a given time period?

In [16]:
# How can buildings be visualised spatially, using colour to indicate aggregated energy usage over a given time period?

q = """
PREFIX s4bldg: <https://saref.etsi.org/saref4bldg/>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
SELECT ?building ?polygon WHERE {
  ?building a s4bldg:Building ;
            geo:hasGeometry ?geometry .
  ?geometry geo:asWKT ?polygon .
}
"""

m = folium.Map(location=[51.985213, 5.870865], zoom_start=17)
bldg_data = defaultdict(tuple)

for bldg_uri, polygon in g.query(q):
    q2 = f"""
    PREFIX saref: <https://saref.etsi.org/core/>
    PREFIX ex: <https://example.org/>

    SELECT ?meter ?time ?value WHERE{{
      ?meter ex:measuresBuilding <{bldg_uri}> .
      ?observation saref:madeBy ?meter ;
                   saref:hasResult ?result ;
                   saref:hasTimestamp ?time .
      ?result saref:hasValue ?value ;
              saref:isMeasuredIn <http://qudt.org/vocab/unit/KiloW-HR> .
    }} 
    ORDER BY ?meter ?time
    """

    result = g.query(q2)
    bldg_usage = -1

    if result:
        bldg_usage = 0
        meter_values = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))      
        _, bldg_name = split_uri(bldg_uri)

        for meter_uri, time, value in result:
            minute = time[:16]
            day = time[:10]
            meter_values[meter_uri][day][minute].append(float(value))

        complete = defaultdict(lambda: defaultdict(list))
        meter_counter = 0
        for meter_uri, days in meter_values.items():
            meter_counter += 1

            for day, times in days.items():
              for time, values in times.items():
                  if len(values) == 4:
                      complete[day][time].append(max(values))
        
        for day, times in complete.items():
            valid_time = []

            for time in sorted(times.keys()):
                if len(times[time]) == meter_counter:
                    valid_time.append(time)
        
            earliest = valid_time[0]
            latest = valid_time[-1]

            bldg_usage += sum(times[latest]) - sum(times[earliest])
                
    geom = wkt.loads(polygon)
    _, bldg_name = split_uri(bldg_uri)

    bldg_data[bldg_name] = (geom, bldg_usage)

colour_map = cm.LinearColormap(colors=["#ffffb2", "#fd8d3c", "#bd0026"], vmin=0, vmax=max(usage for _, usage in bldg_data.values()))

for name, (location, numbers) in bldg_data.items():
    if numbers == -1:
        colour = "#cccccc"
        message = f"{name} has no meters."
    else:
        colour = colour_map(numbers)
        message = f"{name}: {numbers} kWh"

    folium.GeoJson(data=mapping(location), style_function=lambda feature, color=colour: {"fillColor":color, "color": "black", "weight": 1, "fillOpacity": 0.7,}, tooltip=message).add_to(m)

colour_map.caption = "Aggregated energy usage per day in kWh"
colour_map.add_to(m)

print("Map with spacial representation of the buildings in the business park, colours representing the aggregated energy usage per building:")
m

Map with spacial representation of the buildings in the business park, colours representing the aggregated energy usage per building:
